In [9]:
!pip install sentencepiece datasets sacrebleu rouge_score py7zr -q


[notice] A new release of pip is available: 23.2 -> 23.2.1
[notice] To update, run: pip install --upgrade pip


In [11]:
from transformers import pipeline, set_seed

import matplotlib.pyplot as plt

import pandas as pd
from datasets import load_dataset, load_metric
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import sent_tokenize

from transformers import pipeline, set_seed

nltk.download("punkt")

[nltk_data] Downloading package punkt to /Users/stevieg/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
import re

def get_chapter(book_name, chapter_number):
    
    filepath = book_name+'.txt'
    with open(filepath, "r", encoding="utf8") as file:
        text = file.read()
        text = text.replace('\n',' ')

    # Brute Force regular expression rules to fit most conditions

    # chapters divided by *  *  *  *  * 1179
    case1 = re.search('\*\s+\*\s+\*\s+\*\s+\*',text)
    #chapters divided by chpater Chapter CHAPTER chapters Chapters CHAPTERS 1845
    case2 = (len(re.findall('chapter',text,flags = re.IGNORECASE)) != 0)
    # both case 1 and case2 is 729


    # case3 = 

    if case1 and case2:
        print('case1&2')
        matchesone=[match.span()[0] for match in re.finditer('chapter',text,flags=re.IGNORECASE)]
        fi_ch = 0
        for i in range(1,len(matchesone)):
            if matchesone[i]-matchesone[i-1]<50:
                fi_ch+=1
            else:
                continue
#         print(first_ch)
        chapters = re.split("chapter", text, flags = re.IGNORECASE)
        num = chapter_number
        return ''.join(chapters[1:num+fi_ch])


    elif case1:
        chapters = re.split("\*\s+\*\s+\*\s+\*\s+\*", text)
        num = chapter_number
        return ''.join(chapters[1:num])


    elif case2:
#         print('case2')
        matchesone=[match.span()[0] for match in re.finditer('chapter',text,flags=re.IGNORECASE)]
        fi_ch = 0
        for i in range(1,len(matchesone)):
            if matchesone[i]-matchesone[i-1]<50:
                fi_ch+=1
            else:
                continue
#         print(first_ch)
        chapters = re.split("chapter", text, flags = re.IGNORECASE)
        num = chapter_number
        return ''.join(chapters[1:num+fi_ch])


    else:
        return None

# result = get_chapter("Wait and Hope; Or, A Plucky Boy's Luck",5)


In [22]:
res = get_chapter("Wait and Hope; Or, A Plucky Boy's Luck", 3)

In [24]:
res = res[:512]

In [27]:
pipe = pipeline("summarization", model="t5-small")

pipe_out = pipe(res)

Your max_length is set to 200, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


In [28]:
pipe_out

[{'summary_text': 'two boys wake in company in milltown . they were nearly of an age, and both looked sober . "it\'s rather hard to get a lot of people out there," he says .'}]

In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

reference = res

records = []

for model_name in summaries:
    rouge_metric.add(prediction = summaries[model_name], reference = reference )
    score = rouge_metric.compute()
    rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )
    print('rouge_dict ', rouge_dict )
    records.append(rouge_dict)

pd.DataFrame.from_records(records, index = summaries.keys() )

In [21]:
rouge_metric = load_metric('rouge')
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
records = []

rouge_metric.add(prediction = pipe_out, reference = res )
score = rouge_metric.compute()
rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )
print('rouge_dict ', rouge_dict )
records.append(rouge_dict)

rouge_dict

rouge_dict  {'rouge1': 0.022162525184687705, 'rouge2': 0.013440860215053762, 'rougeL': 0.019476158495634655, 'rougeLsum': 0.019476158495634655}


{'rouge1': 0.022162525184687705,
 'rouge2': 0.013440860215053762,
 'rougeL': 0.019476158495634655,
 'rougeLsum': 0.019476158495634655}